# Dimakopoulos et al. (2022) on OpenNeuro ds004752

Paper: Dimakopoulos V, Mégevand P, Stieglitz LH, Imbach L, Sarnthein J (2022). Information flows
from hippocampus to auditory cortex during replay of verbal working memory items. eLife 11:e78677,
[doi:10.7554/elife.78677](https://doi.org/10.7554/elife.78677).

Dataset: [OpenNeuro ds004752](https://openneuro.org/datasets/ds004752/versions/1.0.1), snapshot
1.0.1 (doi:10.18112/openneuro.ds004752.v1.0.1). The claims file names this snapshot. It is also the
latest one: 1.0.0 (2023-09-13) held only the iEEG and EEG, and 1.0.1 (2023-09-20) added the LCMV
beamforming sources that the EEG-source claims need (the dataset's CHANGES file), so 1.0.1 is the
version to pin. Every file is streamed from its pinned S3 object version; the ids of the files read
are printed below.

Claims: `claims/openneuro-ds004752/10.7554-elife.78677.yaml`, written from the version of record
(Europe PMC full text) and the paper's code (github.com/vdimakopoulos/verbal_working_memory at
395ae13).

| Claim | Tested here |
|---|---|
| cohort-sessions | tested |
| fig1b-accuracy | tested |
| capacity-cowan-k | tested |
| fig1c-response-time | tested |
| fig2bc-gamma-encoding-p1 | tested |
| fig2d-cortical-beta-maintenance-p1 | tested |
| hipp-beta-maintenance | tested |
| fig2fg-plv-theta-p1 | tested |
| fig2h-gc-reversal-p1 | tested |
| fig2ijk-spatial-spread-p1 | tested |
| fig3-gc-reversal-p2-p3 | tested |
| fig3no-p4-right-no-gc | tested |
| table1-left-hemisphere-sources | blocked: not runnable (FieldTrip source statistics with an interactive electrode alignment) |
| fig4a-group-gc-reversal | tested |
| scalp-temporal-laplacian | tested |
| fig4bc-auditory-strongest | tested |
| fig4d-reversal-all | tested |
| table1-heschl-dgranger-correct | tested |
| fig4e-lpfc-no-reversal | tested |
| fig4f-incorrect-no-reversal | blocked: data missing (the sources are deposited for correct trials only) |
| balanced-correct-subset | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import wmflow

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("dataset", wmflow.DATASET, "snapshot", wmflow.TAG)
fl = wmflow.files()
print(len(fl), "files,", round(fl["size"].sum() / 1e9, 2), "GB")

## Data check

The dataset's metadata, then the files the analysis uses: the events of every session, the
headers of every EDF file, the electrodes, and the derivative LCMV files. The trial windows are
taken as 8 s epochs from each trial's onset with the probe at 6 s (fixation at -6 s), and this
is checked below against the recordings.

In [ ]:
desc = wmflow.read_json("dataset_description.json")
print(desc["Name"]); print("ReferencesAndLinks:", desc["ReferencesAndLinks"])
print(pd.Series(wmflow.read_json("participants.json")).map(lambda d: d.get("Description")).to_string())
part = wmflow.read_tsv("participants.tsv")
print(part.to_string())
ss = wmflow.sessions()
ev = wmflow.events("ieeg")
ev_eeg = wmflow.events("eeg")
print("\nevents columns:", list(ev.columns))
print("sessions:", len(ss), " subjects:", ss["sub"].nunique(), " trials:", len(ev))
per_sub = ev.groupby("sub").agg(sessions=("ses", "nunique"), trials=("nTrial", "size"))
per_ses = ev.groupby(["sub", "ses"]).size()
print(per_sub.T.to_string())
print("trials per session:", per_ses.value_counts().sort_index().to_dict())
cols = ["sub", "ses", "nTrial", "SetSize", "ProbeLetter", "Match", "Correct", "ResponseTime", "Artifact"]
same = (ev[cols].fillna("n/a").astype(str).values == ev_eeg[cols].fillna("n/a").astype(str).values).all(0)
print("iEEG and EEG events tables agree, column by column:", dict(zip(cols, same.tolist())))
print("ProbeLetter missing (n/a) by subject:", ev[ev.ProbeLetter.isna()]["sub"].value_counts().sort_index().to_dict())
for c in ["SetSize", "Match", "Correct", "Artifact"]:
    print(c, ev[c].value_counts(dropna=False).sort_index().to_dict())
print("ResponseTime (s): min %.3f median %.3f max %.3f" % (ev.ResponseTime.min(), ev.ResponseTime.median(), ev.ResponseTime.max()))
print("duration (s):", ev.duration.unique(), " onset step within session (s):",
      np.unique(np.round(ev.groupby(["sub", "ses"]).onset.diff().dropna(), 4)))
bad = ev[ev.SetSize == 0]
print("\ntrials with SetSize 0:", len(bad), "; all have Match n/a, Correct 0, ResponseTime 0, Artifact 1:",
      bool((bad.Match.isna() & (bad.Correct == 0) & (bad.ResponseTime == 0) & (bad.Artifact == 1)).all()),
      "; by subject", bad["sub"].value_counts().to_dict())

In [ ]:
# EDF headers of every recording, against the channels.tsv and sidecar JSON
def head_row(job):
    r, mod, stem = job
    H = wmflow.edf_header(stem + mod + ".edf")
    ch = wmflow.read_tsv(stem + "channels.tsv")
    return dict(sub=r.sub, ses=r.ses, mod=mod, n_ch=len(H["labels"]), fs=int(H["nsamp"][0]),
                          records=H["nrec"], record_s=H["dur"], units=",".join(sorted(set(H["dim"]))) or "(blank)",
                          tsv_fs=",".join(map(str, sorted(ch.sampling_frequency.unique()))),
                          tsv_low=",".join(map(str, sorted(ch.low_cutoff.unique()))),
                          tsv_high=",".join(map(str, sorted(ch.high_cutoff.unique()))),
                          labels_match=[wmflow.strip(x) for x in H["labels"]] == [wmflow.strip(x) for x in ch.name],
                          trials=int((ev["sub"].eq(r.sub) & ev.ses.eq(r.ses)).sum()))

heads = pd.DataFrame(wmflow._pmap(head_row, [(r, m, st) for r in ss.itertuples() for m, st in [("ieeg", r.ieeg), ("eeg", r.eeg)]]))
heads["records_eq_8_per_trial"] = heads.records == 8 * heads.trials
print(heads.groupby(["sub", "mod"]).agg(n_ch=("n_ch", lambda x: sorted(set(x))), fs=("fs", lambda x: sorted(set(x))),
                                         tsv_fs=("tsv_fs", lambda x: sorted(set(x))), record_s=("record_s", "first"),
                                         units=("units", "first"), labels_match=("labels_match", "all"),
                                         records_ok=("records_eq_8_per_trial", "all")).to_string())
print("\nchannels.tsv cutoffs (low_cutoff, high_cutoff):", heads.groupby(["tsv_low", "tsv_high"]).size().to_dict())
for sub in ["sub-01", "sub-14"]:
    j = wmflow.read_json(ss[ss["sub"] == sub].ieeg.iloc[0] + "ieeg.json")
    print(sub, "ieeg.json SamplingFrequency", j["SamplingFrequency"], "| HardwareFilters", j["HardwareFilters"],
          "| iEEGReference", j["iEEGReference"])

In [ ]:
# Electrodes: depth (1.3 mm) and ECoG (4 mm2) contacts, and the hippocampal readings
el_rows, hc_rows = [], []
ELEC = {}
for sub in sorted(ss["sub"].unique()):
    e = wmflow.electrodes(sub)
    ELEC[sub] = e
    hc = wmflow.hippocampal_channels(sub)
    hemi_name = e.label.str.extract(r"([LR])\d+$")[0]
    hemi_atlas = e.AnatomicalLocation.str.extract(r"(Left|Right)")[0].str[0]
    hemi_x = np.where(e.x < 0, "L", "R")
    known = hemi_atlas.notna()
    el_rows.append(dict(sub=sub, contacts=len(e), depth=int((~e.ecog).sum()), ecog=int(e.ecog.sum()),
                        ecog_electrodes=",".join(sorted(e[e.ecog].electrode.unique())),
                        hipp_labelled=int(e.AnatomicalLocation.astype(str).str.contains("Hipp").sum()),
                        atlas_hemi_vs_x_disagree=int((hemi_atlas[known] != hemi_x[known]).sum()),
                        atlas_labelled=int(known.sum()),
                        name_hemi_vs_x_disagree=int((hemi_name.notna() & (hemi_name != hemi_x)).sum())))
    hc_rows.append(dict(sub=sub, electrodes=",".join(hc["electrodes"]), h1=hc["h1"], h2="-".join(hc["h2"]),
                        h3=",".join(hc["h3"]), ref=hc["ref"], ecog_ref=hc["ecog_ref"]))
print(pd.DataFrame(el_rows).to_string())
print(pd.DataFrame(hc_rows).to_string())

Two checks of the electrode table: the hemisphere in the atlas label against the sign of the MNI
x coordinate, and the hemisphere letter of the contact name against the same sign. Counts above.
The first lists disagreements in most subjects (for example sub-01 `ECL1`, x = -24, "Right
Amygdala"), so the atlas labels' hemisphere is not used here; contacts are chosen by name.

### Loading the recordings

For each subject: [-6, 2) s around the probe of every trial of set size 6 or 8, at 500 Hz
(Methods: "All signals were downsampled to 500 Hz"), for the hippocampal contacts (all contacts
in the four ECoG participants), and the derivative LCMV file. This streams about 2.5 GB.

In [ ]:
import time
t0 = time.time()
ALL = {}
for sub in sorted(ss["sub"].unique()):
    ALL[sub] = wmflow.load_subject(sub, ev)
print("loaded in %.0f s" % (time.time() - t0))

In [ ]:
rows = []
for sub, S in ALL.items():
    e = ev[ev["sub"] == sub]
    L = S["lcmv"]
    n_sel = int(wmflow.gc_trials(e).sum())
    n_corr68 = int((e.SetSize.isin([6, 8]) & (e.Correct == 1)).sum())
    rows.append(dict(sub=sub, deriv_trials=L["enc"]["data"].shape[0], gc_trials=n_sel, correct_ss68_any_artifact=n_corr68,
                     deriv_labels=len(L["enc"]["label"]), areas=all(a in L["enc"]["label"] for a in wmflow.AREAS),
                     deriv_fs=L["enc"]["fs"], enc_t=(round(L["enc"]["time"][0], 4), round(L["enc"]["time"][-1], 4)),
                     maint_t=(round(L["maint"]["time"][0], 4), round(L["maint"]["time"][-1], 4)),
                     ieeg_trials_read=S["ieeg"]["data"].shape[0], ieeg_channels_read=S["ieeg"]["data"].shape[1]))
deriv = pd.DataFrame(rows)
print(deriv.to_string())
print("derivative files hold exactly the GC method's trials (correct, set size 6/8, no artifact) in",
      int((deriv.deriv_trials == deriv.gc_trials).sum()), "subjects; the correct set size 6/8 trials with artifacts in",
      int(((deriv.deriv_trials != deriv.gc_trials) & (deriv.deriv_trials == deriv.correct_ss68_any_artifact)).sum()))
print("virtual sensor labels:", ALL["sub-01"]["lcmv"]["enc"]["label"][:5], "; scalp labels per subject:",
      {s: len(S["lcmv"]["enc"]["label"]) - 5 for s, S in ALL.items()})
for per in ["enc", "maint"]:
    v = np.concatenate([S["lcmv"][per]["data"][:, :5].ravel() for S in ALL.values()])
    print(per, "virtual sensor values: median |x| %.3g, 1-99%% range [%.3g, %.3g] (no unit given)" % (
        np.median(np.abs(v)), *np.percentile(v, [1, 99])))

The derivative files carry no trial identifiers. They are matched to the events in order: the
GC method's trials (correct, set size 6 or 8, no artifact flag), or for the two subjects where
the counts say so, all correct trials of set size 6 or 8. The match is checked against the raw
scalp EEG: each derivative trial's scalp channels (encoding window) are correlated with the raw
EEG of every candidate trial of the same subject, re-referenced to the mastoids, and the best
candidate should be the one the order assigns.

In [ ]:
def match_check(sub, S, max_trials=40):
    L = S["lcmv"]["enc"]
    dr = S["deriv_rows"]
    cand = dr  # candidate trials: all derivative rows of the subject
    st = ss.set_index(["sub", "ses"])
    heads_eeg = {s: wmflow.edf_header(st.loc[(sub, s), "eeg"] + "eeg.edf") for s in cand.ses.unique()}
    labs = [l for l in L["label"][5:] if l not in ("A1", "A2")]
    def rd(r):
        H = heads_eeg[r.ses]
        names = [wmflow.strip(x) for x in H["labels"]]
        ci = [names.index(l) for l in labs] + [names.index("A1"), names.index("A2")]
        x = wmflow.edf_records(H, (r.nTrial - 1) * 8 + 1, 2, ci)  # [-5, -3) s
        x = x[:-2] - x[-2:].mean(0)
        return wmflow.to_fs(x, int(H["nsamp"][0]), L["fs"])
    raw = np.stack(wmflow._pmap(rd, list(cand.itertuples())))
    n = min(raw.shape[-1], L["data"].shape[-1])
    li = [L["label"].index(l) for l in labs]
    hits = []
    for j in range(min(max_trials, len(dr))):
        d = L["data"][j, li, :n]
        d = d - d.mean(-1, keepdims=True)
        rr = raw[:, :, :n] - raw[:, :, :n].mean(-1, keepdims=True)
        c = (d[None] * rr).sum(-1) / np.sqrt((d[None] ** 2).sum(-1) * (rr ** 2).sum(-1))
        score = np.nanmean(c, 1)
        hits.append((int(np.argmax(score)) == j, float(score[j]), float(np.median(np.delete(score, j)))))
    h = np.array(hits)
    return dict(sub=sub, checked=len(h), best_is_assigned=int(h[:, 0].sum()), r_assigned=round(np.median(h[:, 1]), 3),
                r_others=round(np.median(h[:, 2]), 3), channels=len(labs))

mc = pd.DataFrame([match_check(sub, S) for sub, S in ALL.items()])
print(mc.to_string())

Clock check of the iEEG. If the trial windows are placed right, the hippocampal contact (reading
h1) should respond to the letters at -5 s. The trial-averaged signal's RMS in the 0.5 s after
-5 s is compared with the 0.5 s before; as a control the same is computed with the onset
placed at -5.5 s and -4.5 s, where nothing happens on the screen at the onset.

In [ ]:
rows = []
for sub, S in ALL.items():
    D = S["ieeg"]
    idx = np.flatnonzero((S["rows"].Correct == 1).values)
    h1 = wmflow.hipp_signals(D["data"][idx].astype(float), D["channels"], S["hc"])["h1"]
    erp = h1.mean(0)
    def ratio(t_on):
        a = (D["time"] >= t_on) & (D["time"] < t_on + 0.5)
        b = (D["time"] >= t_on - 0.5) & (D["time"] < t_on)
        return np.sqrt(np.mean((erp[a] - erp[b].mean()) ** 2)) / np.std(h1[:, b].mean(0) - erp[b].mean() + 1e-12)
    rows.append(dict(sub=sub, trials=len(idx), onset_minus5=ratio(-5.0), control_minus5_5=ratio(-5.5), control_minus4_5=ratio(-4.5),
                     probe_0=ratio(0.0)))
clk = pd.DataFrame(rows).round(2)
print(clk.to_string())
print("subjects where the -5 s onset ratio exceeds both controls:",
      int(((clk.onset_minus5 > clk.control_minus5_5) & (clk.onset_minus5 > clk.control_minus4_5)).sum()), "of", len(clk))

What the checks show. The sampling rates in channels.tsv match the EDF headers in every file,
each recording has 8 one-second records per trial, and the iEEG and EEG events tables agree.
Three points disagree with the description: channels.tsv gives low_cutoff 1000 Hz and
high_cutoff 0.5 Hz, the reverse of the 0.5 to 1000 Hz passband in the sidecars and the paper;
the EDF headers leave the physical unit blank (channels.tsv says µV); and the electrode table's
hemisphere letters disagree with the MNI x coordinate for 16 contacts of sub-11 (PHL at
positive x, PHR at negative x) and 3 of sub-12, and the atlas labels name the other hemisphere
for 12 contacts of sub-01. The derivative files' trials match the raw EEG of the trials assigned
to them in all 40 trials checked per subject (r = 0.70 to 1.00 against 0.04 to 0.31 for other
trials), so the derivatives, the EEG and the events share one clock. For the iEEG, the
hippocampal response after the letters appear at -5 s exceeds both controls in 11 of 15
subjects; in sub-02, sub-03, sub-11 and sub-14 this check does not establish the iEEG timing,
which is taken there as described (8 s epochs from each trial's onset), not shown.

## cohort-sessions

> Fifteen participants (median age 29 years, range 18 to 56, 7 male) performed the modified
> Sternberg task in 71 sessions of 50 trials each.

In [ ]:
n_sub = part.participant_id.nunique()
n_ses = len(ss)
males = int((part.sex.str.lower() == "m").sum())
print("participants", n_sub, "| sessions", n_ses, "| median age", part.age.median(), "| age range", part.age.min(), "-", part.age.max(),
      "| male", males, "| female", int((part.sex.str.lower() == "f").sum()))
print("trials per session:", per_ses.value_counts().sort_index().to_dict(), "| sessions with 50 trials:", int((per_ses == 50).sum()),
      "| total trials", len(ev), "(71 x 50 = 3550 in the paper)")
print("sessions per subject:", per_sub.sessions.to_dict())

**Result: partially reproduced.** 15 participants, median age 29 years, range 18 to 56, and 7
male match the paper. The dataset holds 68 sessions, not 71, and 15 of them have fewer than 50
trials (34 to 49); 3353 trials in all.

## fig1b-accuracy

> The mean correct response rate was 91% (for both IN and OUT trials) and decreased with set
> size, from 97% at set size 4 to 89% at 6 and 83% at 8.

All trials before artifact rejection, as the criterion says. The 17 placeholder trials with set
size 0 (no probe, no response) are left out of every rate; including them changes the pooled
rate by less than 0.5 point (printed).

In [ ]:
beh = ev[ev.SetSize > 0].copy()
def rates(df):
    return pd.Series({"all": df.Correct.mean() * 100, "IN": df[df.Match == "IN"].Correct.mean() * 100,
                      "OUT": df[df.Match == "OUT"].Correct.mean() * 100,
                      **{f"ss{k}": df[df.SetSize == k].Correct.mean() * 100 for k in (4, 6, 8)}})
acc = pd.DataFrame({"a_pooled": rates(beh),
                    "b_mean_of_sessions": beh.groupby(["sub", "ses"]).apply(rates).mean(),
                    "c_mean_of_participants": beh.groupby("sub").apply(rates).mean()}).T.round(1)
print(acc.to_string())
print("pooled rate with the 17 set size 0 trials counted as errors: %.1f%%" % (ev.Correct.mean() * 100))
exp = {"all": 91, "ss4": 97, "ss6": 89, "ss8": 83}
for rd, r in acc.iterrows():
    ok = all(abs(r[k] - v) <= 0.1 * v for k, v in exp.items())
    dec = r.ss4 > r.ss6 > r.ss8
    print(rd, "all four within 10%:", ok, "| decreasing 4 > 6 > 8:", dec)
print("by participant:\n", beh.groupby("sub").apply(rates).round(1).T.to_string())

**Result: reproduced.** Under each of the three readings the overall rate and the rates at set
sizes 4, 6 and 8 are within 10% of 91, 97, 89 and 83%, and the rate falls from 4 to 6 to 8
(pooled: 91.3, 97.7, 90.3 and 84.5%). The paper's "both for IN and OUT" does not hold exactly:
pooled IN trials are 89.0% correct and OUT trials 93.5%.

## capacity-cowan-k

> Across participants the memory capacity (Cowan's K) averaged 6.1.

K = (correct IN rate + correct OUT rate - 1) x set size, readings (a) to (d) of the criterion.

In [ ]:
def cowan(df, from_ss8_only):
    d = df[df.SetSize == 8] if from_ss8_only else df
    hit = d[d.Match == "IN"].Correct.mean()
    cr = d[d.Match == "OUT"].Correct.mean()
    return (hit + cr - 1) * 8
rows = []
for unit, keys in [("participants", ["sub"]), ("sessions", ["sub", "ses"])]:
    for src, ss8 in [("rates over all trials (a, b)", False), ("rates at set size 8 (c)", True)]:
        k = beh.groupby(keys).apply(lambda d: cowan(d, ss8))
        rows.append(dict(unit=unit, rates=src, mean=k.mean(), median=k.median(), min=k.min(), max=k.max(), n=len(k)))
ck = pd.DataFrame(rows).round(2)
print(ck.to_string())
print("readings within 6.1 +- 0.61:", int(((ck[["mean", "median"]] - 6.1).abs() <= 0.61).values.sum()), "of", ck[["mean", "median"]].size)

**Result: reproduced.** Reading (a), rates over all trials times set size 8, averaged over
participants, gives K = 6.59, within 6.1 +- 0.61, and so does its median (reading b, 6.63) and
the same over sessions (6.57 and 6.67). K from the set size 8 rates alone (reading c) is 5.43
(mean) and 5.33 (median), outside.

## fig1c-response-time

> The response time of correct trials (3045 trials) was 1.1 +- 0.5 s and grew with set size
> (1.1, 1.2 and 1.3 s for 4, 6 and 8 letters, 53 ms per item); correct decisions were faster than
> incorrect ones (1.1 against 1.3 s).

In [ ]:
rows = []
for pool, d in [("p1 all correct", beh[beh.Correct == 1]), ("p2 correct, no artifact", beh[(beh.Correct == 1) & (beh.Artifact == 0)])]:
    inc = beh[beh.Correct == 0]
    slope = stats.linregress(d.SetSize, d.ResponseTime).slope * 1000
    for c, fn in [("a mean", np.mean), ("b median", np.median)]:
        rows.append(dict(pool=pool, central=c, n=len(d), all=fn(d.ResponseTime), sd=d.ResponseTime.std(),
                         ss4=fn(d[d.SetSize == 4].ResponseTime), ss6=fn(d[d.SetSize == 6].ResponseTime),
                         ss8=fn(d[d.SetSize == 8].ResponseTime), slope_ms_per_item=slope,
                         incorrect=fn(inc.ResponseTime), incorrect_sd=inc.ResponseTime.std(), n_incorrect=len(inc)))
rt = pd.DataFrame(rows).round(3)
print(rt.to_string())
exp = dict(all=1.1, ss4=1.1, ss6=1.2, ss8=1.3, slope_ms_per_item=53, incorrect=1.3)
for _, r in rt.iterrows():
    within = {k: abs(r[k] - v) <= 0.1 * v for k, v in exp.items()}
    print(r.pool, "|", r.central, "| within 10%:", within, "| correct faster:", r["all"] < r.incorrect)
print("correct trials in the events:", int((ev.Correct == 1).sum()), "| response times > 5 s:", int((beh.ResponseTime > 5).sum()))

**Result: reproduced.** The events hold exactly 3045 correct trials, the paper's count. Under
reading (b), the median, with pool p2 (correct trials without the artifact flag, 2903), every
value is within 10%: 1.13 s overall, 1.07, 1.14 and 1.21 s at set sizes 4, 6 and 8, a slope of
48.1 ms per item (tolerance 47.7 to 58.3) and 1.39 s for incorrect trials; correct trials are
faster. With pool p1 (all 3045) the slope is 47.69 ms per item, just outside, and the rest
holds. Under reading (a), the mean, the overall (1.29 to 1.30 s) and incorrect (1.70 s) values
are outside. The spread is 0.58 to 0.60 s against the paper's 0.5.

## The four ECoG participants

Participants 1 to 4 are sub-10 to sub-13 (participants.tsv, `eLife_identifier`), and they are the
only subjects with ECoG contacts (4 mm2) in the electrode tables: sub-10 a grid GL_A1 to GL_H8
(columns A to H, rows 1 to 8, as Figure 2a describes), sub-11 three left strips, sub-12 one strip
(PL), sub-13 two right strips (GAR, GSR). ECoG is re-referenced to a white-matter depth contact
other than the hippocampal reference (Methods, Recording setup; the contacts are printed in the
data check). Trials: the GC method's (correct, set size 6 or 8, no artifact flag) unless a
reading says otherwise.

In [ ]:
P = {sub: wmflow.ECOG_SUBJECTS[sub] for sub in wmflow.ECOG_SUBJECTS}
def trial_idx(S, correct_only=True):
    r = S["rows"]
    m = (r.Artifact == 0) & ((r.Correct == 1) if correct_only else True)
    return np.flatnonzero(m.values)
GRID = [f"GL_{c}{k}" for c in "ABCDEFGH" for k in range(1, 9)]
S1 = ALL["sub-10"]
el1 = S1["electrodes"].set_index("label")
def region(lab):
    a = str(el1.loc[lab, "AnatomicalLocation"])
    for key, name in [("Temporal", "temporal"), ("STS", "temporal"), ("Occipital", "occipital"), ("Parietal", "parietal"),
                      ("Postcentral", "parietal"), ("Hipp", "hippocampus label"), ("no_label", "no label")]:
        if key in a:
            return name
    return "other"
REGION = {c: region(c) for c in GRID}
AUDITORY = ["GL_C2", "GL_C3", "GL_B1", "GL_B2", "GL_B3", "GL_C1", "GL_D1", "GL_D2", "GL_D3"]
OCCIPITAL = ["GL_H3"]
print("grid contacts by atlas region:", pd.Series(REGION).value_counts().to_dict())
for sub in P:
    S = ALL[sub]
    print(sub, "Participant", P[sub], "| GC-method trials", len(trial_idx(S)), "| all set size 6/8 trials without artifact",
          len(trial_idx(S, False)), "| ECoG contacts", int(S["electrodes"].ecog.sum()), "| ECoG reference", S["hc"]["ecog_ref"])

def grid_print(values, fmt="{:7.2f}"):
    """Print a grid map: rows 8 (top) to 1, columns A to H."""
    v = pd.Series(values)
    tab = pd.DataFrame({c: [v.get(f"GL_{c}{k}", np.nan) for k in range(8, 0, -1)] for c in "ABCDEFGH"}, index=range(8, 0, -1))
    print(tab.to_string(float_format=lambda x: fmt.format(x)))

## fig2bc-gamma-encoding-p1

> In Participant 1, gamma power (above 40 Hz) rose by more than 100% over fixation during encoding
> in an occipital contact (H3) and a temporal contact over auditory cortex (C2), and gamma
> increased significantly only in temporal and occipital-parietal contacts.

Relative PSD as the Methods give it, with readings (a) correct trials and (b) all trials of set
sizes 6 and 8. Then the code's z-score test of the grid map (PSD_fix_vs_enc_Permutation_test.m).

In [ ]:
FREQ_B = np.arange(9.0, 25.0, 1.0)
FREQ_G = np.arange(40.0, 101.0, 2.0)
E1 = {rd: wmflow.ecog_signals(S1, trial_idx(S1, rd == "a")) for rd in ["a", "b"]}
TF1 = {}
for rd in ["a", "b"]:
    X = np.stack([E1[rd][c] for c in GRID], 1)
    for nm, fr in [("gamma", FREQ_G), ("beta", FREQ_B)]:
        Pw, tt = wmflow.tfr_power(X, 500, fr, S1["ieeg"]["time"])
        TF1[(rd, nm)] = (Pw, tt, wmflow.relative(Pw, fr, tt))
rows = []
for rd in ["a", "b"]:
    Pw, tt, R = TF1[(rd, "gamma")]
    g = dict(zip(GRID, wmflow.tf_mean(R, FREQ_G, tt, (60, 80), (-3.5, -3.0))))
    for c, band, win in [("GL_H3", (40, 100), (-5, -3)), ("GL_C2", (60, 80), (-4, -3))]:
        i = GRID.index(c)
        rows.append(dict(reading=rd, contact=c, band=band, window=win, trials=Pw.shape[0],
                         mean_increase_pct=100 * wmflow.tf_mean(R[i], FREQ_G, tt, band, win),
                         max_increase_pct=100 * wmflow.tf_mean(R[i], FREQ_G, tt, band, win, "max")))
    if rd == "a":
        print("relative gamma [60 80] Hz in [-3.5 -3] s, %, reading a (Figure 2a map); range",
              round(100 * min(g.values()), 1), "to", round(100 * max(g.values()), 1))
        grid_print({k: 100 * v for k, v in g.items()}, "{:7.0f}")
print(pd.DataFrame(rows).round(1).to_string())

In [ ]:
from scipy.signal.windows import dpss
def psd_ztest(x_fix, x_enc, rng, n_perm=200, fs=500):
    """The code's test: mtmfft power (1 dpss taper, 2 Hz smoothing over 0.5 s), trials permuted
    between the two periods, (enc - fix) minus the null 95th percentile, z-scored over 60-100 Hz,
    maximum taken."""
    n = x_fix.shape[-1]
    tap = dpss(n, 1.0, 1)[0]
    def pw(x):
        x = x - x.mean(-1, keepdims=True)
        return np.abs(np.fft.rfft(x * tap, axis=-1)) ** 2
    pf, pe = pw(x_fix), pw(x_enc)
    f = np.fft.rfftfreq(n, 1 / fs)
    obs = pe.mean(0) - pf.mean(0)
    pool = np.concatenate([pf, pe])
    nf = len(pf)
    null = np.empty((n_perm, len(f)))
    for i in range(n_perm):
        p = rng.permutation(len(pool))
        null[i] = pool[p[nf:]].mean(0) - pool[p[:nf]].mean(0)
    sur = obs - np.percentile(null, 95, axis=0)
    band = (f >= 60) & (f <= 100)
    z = stats.zscore(sur[band])
    # the same statistic for one permuted data set, to show what it gives without an effect
    p = rng.permutation(len(pool))
    sur0 = (pool[p[nf:]].mean(0) - pool[p[:nf]].mean(0)) - np.percentile(null, 95, axis=0)
    return z.max(), stats.zscore(sur0[band]).max(), (obs[band] > np.percentile(null, 95, axis=0)[band]).mean()
rng = np.random.default_rng(21)
t = S1["ieeg"]["time"]
fixw = (t >= -5.5) & (t < -5.0); encw = (t >= -3.5) & (t < -3.0)
zt = {c: psd_ztest(E1["a"][c][:, fixw], E1["a"][c][:, encw], rng) for c in GRID}
zmax = {c: v[0] for c, v in zt.items()}
print("max z over 60-100 Hz per contact (significant > 1.96); range", round(min(zmax.values()), 2), "to", round(max(zmax.values()), 2))
grid_print(zmax)
sig = [c for c in GRID if zmax[c] > 1.96]
print("contacts with z > 1.96:", len(sig), "of 64; by region", pd.Series([REGION[c] for c in sig]).value_counts().to_dict())
print("same statistic on one trial-permuted data set per contact: > 1.96 in", sum(v[1] > 1.96 for v in zt.values()), "of 64 contacts")
frac = {c: v[2] for c, v in zt.items()}
print("fraction of 60-100 Hz bins where encoding exceeds the null 95th percentile; contacts with any:",
      sum(v > 0 for v in frac.values()), "; with more than half:", sum(v > 0.5 for v in frac.values()),
      "by region", pd.Series([REGION[c] for c in GRID if frac[c] > 0.5]).value_counts().to_dict())

**Result: partially reproduced.** Gamma at H3 rose by 179% over [40 100] Hz and [-5 -3] s (183%
with all trials, reading b), above 100%. At C2, gamma [60 80] Hz in [-4 -3] s rose by 17% on average
(14% with all trials; its largest time-frequency point is 61%), not above 100%. The code's z-score
test marks 11 of 64 contacts (5 temporal, 3 parietal, 1 occipital, 2 without an atlas label), so
the significant contacts do lie in temporal and occipital-parietal cortex. The statistic is
z-scored over frequencies and its maximum taken, which can exceed 1.96 without an effect: on one
trial-permuted data set it did so at 1 contact of 64.

## fig2d-cortical-beta-maintenance-p1

> In Participant 1, high alpha to low beta activity ([11 14] Hz) appeared toward the end of
> maintenance in temporal and occipital contacts.

Relative PSD over [11 14] Hz and [-2 0] s. Per contact, a permutation test of maintenance
against fixation: in each of 200 permutations the two periods are swapped in a random half of
the trials; one-sided p for an increase.

In [ ]:
rng = np.random.default_rng(22)
out = {}
for rd in ["a", "b"]:
    Pw, tt, R = TF1[(rd, "beta")]
    rel = dict(zip(GRID, 100 * wmflow.tf_mean(R, FREQ_B, tt, (11, 14), (-2, 0))))
    fm = (FREQ_B >= 11) & (FREQ_B <= 14)
    bm = wmflow.baseline_mask(FREQ_B, tt)[fm].all(0)
    mt = (tt >= -2) & (tt < 0)
    m_tr = Pw[:, :, fm][..., mt].mean((-1, -2))
    f_tr = Pw[:, :, fm][..., bm].mean((-1, -2))
    obs = m_tr.mean(0) / f_tr.mean(0) - 1
    null = np.empty((200, len(GRID)))
    for i in range(200):
        sw = rng.random(len(m_tr))[:, None] < 0.5
        a_, b_ = np.where(sw, f_tr, m_tr), np.where(sw, m_tr, f_tr)
        null[i] = a_.mean(0) / b_.mean(0) - 1
    p = (1 + (null >= obs[None]).sum(0)) / 201
    out[rd] = pd.DataFrame(dict(rel_pct=pd.Series(rel), p=pd.Series(p, index=GRID), region=pd.Series(REGION)))
for rd, df in out.items():
    sigc = df[df.p < 0.05]
    top = df.sort_values("rel_pct", ascending=False).head(8)
    print(f"reading {rd}: relative [11 14] Hz power in [-2 0] s, range {df.rel_pct.min():.0f} to {df.rel_pct.max():.0f}%;",
          "significant contacts:", len(sigc), sigc.region.value_counts().to_dict())
    print("  eight largest:", [(c, round(v, 0), REGION[c]) for c, v in top.rel_pct.items()])
grid_print(out["a"].rel_pct, "{:7.0f}")

**Result: reproduced.** The largest [11 14] Hz increases in [-2 0] s are at the posterior contacts
H3 (35%), H4 (26%, temporal), H2, F8 (occipital), H5 and G3 (occipital), and the 5 contacts with
a significant increase over fixation (p < 0.05) are 2 temporal, 1 occipital and 2 without an
atlas label (H3, which the paper calls occipital, has no atlas label). With all trials (reading b) the
pattern is the same (6 significant contacts). The anterior and inferior grid shows decreases
(down to -53%).

## hipp-beta-maintenance

> In the hippocampus of all four participants with ECoG, beta activity ([12 24] Hz) rose by more
> than 100% toward the end of maintenance.

In [ ]:
BANDS2 = {"sub-10": (12, 24), "sub-11": (9, 18), "sub-12": (13, 21), "sub-13": (13, 21)}
rows = []
for sub in P:
    S = ALL[sub]
    idx = trial_idx(S)
    hs = wmflow.hipp_signals(S["ieeg"]["data"][idx].astype(float), S["ieeg"]["channels"], S["hc"])
    for h in ["h1", "h2", "h3"]:
        xs = hs[h] if h == "h3" else [hs[h]]
        Pw, tt = wmflow.tfr_power(np.stack(xs, 1), 500, FREQ_B, S["ieeg"]["time"])
        R = wmflow.relative(Pw, FREQ_B, tt)
        for bname, band in [("12-24", (12, 24)), ("legend", BANDS2[sub])]:
            for wname, win in [("w1 [-1 0]", (-1, 0)), ("w2 [-2 0]", (-2, 0))]:
                for how in ["mean", "max"]:
                    v = wmflow.tf_mean(R, FREQ_B, tt, band, win, how)
                    rows.append(dict(sub=sub, participant=P[sub], h=h, band=bname, window=wname, measure=how,
                                     increase_pct=100 * float(np.mean(v))))
hb = pd.DataFrame(rows)
tab = hb.pivot_table(index=["band", "window", "measure", "h"], columns="participant", values="increase_pct").round(0)
tab["n_over_100"] = (tab > 100).sum(axis=1)
print(tab.to_string())
print("readings with all four participants over 100%:", int((tab.n_over_100 == 4).sum()), "of", len(tab),
      "| with three:", int((tab.n_over_100 == 3).sum()))

**Result: not reproduced.** No reading gives an increase over 100% in all four participants, nor
in three. The band means stay below 100% in every participant (largest 94%, Participant 3, h1,
[12 24] Hz, [-2 0] s); with the maximum over time-frequency points, Participant 3 (h1, up to
250%) and Participant 4 (h3, up to 209%) exceed 100%, Participant 1 reaches 55% and Participant 2
at most 12%.

## fig2fg-plv-theta-p1

> In Participant 1, phase locking between hippocampus and auditory cortex was enhanced in the theta
> range during maintenance (PLV 0.4 at contact C3), and theta PLV in maintenance was highest to
> contacts over auditory cortex.

PLV from two DPSS tapers at 1 Hz resolution (maintenance [-2 0] s, fixation [-6 -5] s, encoding
[-5 -3] s as the variant). Per contact, the null swaps the two periods' phasors in a random half
of the trials (200 permutations); the theta value is the mean over 4 to 8 Hz.

In [ ]:
def phasor_set(x, win, t, fs=500):
    m = (t >= win[0]) & (t < win[1])
    F, f = wmflow.multitaper_fourier(x[:, m], fs, 2)
    keep = (f >= 1) & (np.abs(f - np.round(f)) < 1e-9) & (f <= 100)
    return F[..., keep], f[keep]
rng = np.random.default_rng(23)
idx = trial_idx(S1)
hs1 = wmflow.hipp_signals(S1["ieeg"]["data"][idx].astype(float), S1["ieeg"]["channels"], S1["hc"])
t = S1["ieeg"]["time"]
plv_rows, plv_spec = [], {}
for h in ["h1", "h2", "h3"]:
    xs = hs1[h] if h == "h3" else [hs1[h]]
    for comp, cwin in [("fixation", wmflow.WINDOWS["fix"]), ("encoding", wmflow.WINDOWS["enc"])]:
        for c in GRID:
            ph_m, ph_c = [], []
            for x in xs:
                Fm_h, f = phasor_set(x, wmflow.WINDOWS["maint"], t); Fm_c, _ = phasor_set(E1["a"][c], wmflow.WINDOWS["maint"], t)
                Fc_h, _ = phasor_set(x, cwin, t); Fc_c, _ = phasor_set(E1["a"][c], cwin, t)
                ph_m.append(wmflow.phasors(Fm_h, Fm_c)); ph_c.append(wmflow.phasors(Fc_h, Fc_c))
            th = (f >= 4) & (f <= 8)
            def plv_of(pm, pc):
                return np.mean([np.abs(a.mean(0)) for a in pm], 0), np.mean([np.abs(b.mean(0)) for b in pc], 0)
            pm_, pc_ = plv_of(ph_m, ph_c)
            obs = pm_ - pc_
            null = np.empty((200, len(f)))
            for i in range(200):
                sw = rng.random(len(idx))[:, None] < 0.5
                a_ = [np.where(sw, b, a) for a, b in zip(ph_m, ph_c)]
                b_ = [np.where(sw, a, b) for a, b in zip(ph_m, ph_c)]
                x1, x2 = plv_of(a_, b_)
                null[i] = x1 - x2
            thr = np.percentile(null, 95, axis=0)
            sigf = obs > thr
            null_th = null[:, th].mean(1)
            plv_rows.append(dict(h=h, comparison=comp, contact=c, region=REGION[c], plv_maint_theta=pm_[th].mean(),
                                 plv_comp_theta=pc_[th].mean(), diff_theta=obs[th].mean(),
                                 p_theta=(1 + (null_th >= obs[th].mean()).sum()) / 201,
                                 sig_bands=wmflow.runs(sigf, f)))
plv = pd.DataFrame(plv_rows)
for (h, comp), d in plv.groupby(["h", "comparison"]):
    d = d.set_index("contact")
    top = d.plv_maint_theta.sort_values(ascending=False).head(6)
    sigc = d[d.p_theta < 0.05]
    print(f"{h} vs {comp}: C3 theta PLV maintenance {d.loc['GL_C3', 'plv_maint_theta']:.3f} ({comp} {d.loc['GL_C3', 'plv_comp_theta']:.3f}, "
          f"p {d.loc['GL_C3', 'p_theta']:.3f}, bands above the null 95th percentile {d.loc['GL_C3', 'sig_bands'][:4]}); "
          f"PLV range {d.plv_maint_theta.min():.3f} to {d.plv_maint_theta.max():.3f}; significant contacts {len(sigc)} "
          f"({sum(c in AUDITORY for c in sigc.index)} auditory); six highest {[(c, round(v, 3)) for c, v in top.items()]}; "
          f"auditory among them {sum(c in AUDITORY for c in top.index)}")
grid_print(plv[(plv.h == "h1") & (plv.comparison == "fixation")].set_index("contact").plv_maint_theta, "{:7.3f}")

**Result: not reproduced.** Theta PLV between the hippocampus and C3 in maintenance is 0.240 (h1),
0.164 (h2) and 0.195 (h3), below 0.4 - 0.04. It does not exceed fixation (p = 0.72, 0.49 and
0.35), and no contact has a significant theta increase over fixation under h1 or h3 (2 contacts,
none auditory, under h2). Against encoding, the variant, C3 is significant (p = 0.010 to 0.025)
and 16 to 21 contacts are, 4 to 6 of them auditory. The highest maintenance theta PLV is at
A1 to A3 (anterior, inferior grid), with C3 and B3 among the six highest.

## fig2h-gc-reversal-p1

> In Participant 1, information flowed from auditory cortex to hippocampus during encoding (net
> flow significant in [6 8] Hz) and reversed, from hippocampus to auditory cortex, during
> maintenance (significant in [5 8] Hz).

The GC method between the hippocampal readings and grid contact C2. Null: GC recomputed 200 times
with the two channels swapped in a random half of the trials (Methods, Statistics, paragraph 1).
Reading c1: each frequency against the null's 5th (encoding) or 95th (maintenance) percentile;
reading c2: contiguous runs of such frequencies as clusters, each tested by its mass against the
largest cluster mass of each null draw. Then the code's variant: bipolar AHL2-AHL3 with C2, null
from 200 shufflings of the trial order between the two channels, 1st or 99th percentile.

In [ ]:
def hipp_reading_signals(S, idx, hc=None):
    """Hippocampal readings at 40 Hz in encoding and maintenance; h3 is a list of channels."""
    hs = wmflow.hipp_signals(S["ieeg"]["data"][idx].astype(float), S["ieeg"]["channels"], hc or S["hc"])
    return {(h, per): [wmflow.window40(x, per) for x in (hs[h] if h == "h3" else [hs[h]])]
            for h in ["h1", "h2", "h3"] for per in ["enc", "maint"]}

def ecog_gc_rows(sub, contacts, n_perm=200, seed=0, hc=None, readings=("h1", "h2", "h3")):
    S = ALL[sub]
    idx = trial_idx(S)
    H = hipp_reading_signals(S, idx, hc)
    E = wmflow.ecog_signals(S, idx)
    rows, spec = [], {}
    for c in contacts:
        for per in ["enc", "maint"]:
            cx = wmflow.window40(E[c], per)
            for h in readings:
                g = wmflow.gc_test(H[(h, per)], cx, n_perm, seed)
                sb = wmflow.significant_bands(g["freq"], g["dg"], g["null"], per)
                th = (g["freq"] >= 4) & (g["freq"] <= 8)
                rows.append(dict(sub=sub, contact=c, period=per, h=h, d1=wmflow.band_value(g["freq"], g["dg"]),
                                 d2=wmflow.band_value(g["freq"], g["dg"], how="ext", period=per),
                                 c1_bands=sb["bands"], c1_theta=any(a <= 8 and b >= 4 for a, b in sb["bands"]),
                                 c2_clusters=[(x["band"], round(x["p"], 3)) for x in sb["clusters"] if x["p"] < 0.05],
                                 c2_theta=any(x["p"] < 0.05 and x["band"][0] <= 8 and x["band"][1] >= 4 for x in sb["clusters"]),
                                 null_d1_5_95=tuple(np.round(np.percentile(g["null"][:, th].mean(1), [5, 95]), 4))))
                spec[(c, per, h)] = g
    return pd.DataFrame(rows), spec

t0 = time.time()
gc_p1, spec_p1 = ecog_gc_rows("sub-10", ["GL_C2"], seed=31)
print("%.0f s" % (time.time() - t0))

In [ ]:
print(gc_p1.drop(columns=["sub"]).to_string())
for h in ["h1", "h2", "h3"]:
    d = gc_p1[gc_p1.h == h].set_index("period")
    print(h, "| c1: encoding negative in theta", d.loc["enc", "c1_theta"], "maintenance positive in theta", d.loc["maint", "c1_theta"],
          "| c2:", d.loc["enc", "c2_theta"], d.loc["maint", "c2_theta"])
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for a, h in zip(ax, ["h1", "h2", "h3"]):
    for per, col in [("enc", "tab:blue"), ("maint", "tab:red")]:
        g = spec_p1[("GL_C2", per, h)]
        sel = (g["freq"] >= 4) & (g["freq"] <= 20)
        a.plot(g["freq"][sel], g["dg"][sel], color=col, label=per)
        a.fill_between(g["freq"][sel], *np.percentile(g["null"][:, sel], [5, 95], axis=0), color=col, alpha=0.2)
    a.axhline(0, color="k", lw=0.5); a.set_title(f"P1 C2, {h}"); a.set_xlabel("Hz")
ax[0].set_ylabel("ΔGranger (hipp→ctx − ctx→hipp)"); ax[0].legend()
allv = np.concatenate([spec_p1[k]["dg"] for k in spec_p1])
print("plotted ΔGranger range %.3f to %.3f, none outside the axes" % (allv.min(), allv.max()))
plt.show()

In [ ]:
# The code's variant (Statistics/Granger_significance_calculation.m)
S = ALL["sub-10"]; idx = trial_idx(S)
c = {n: i for i, n in enumerate(S["ieeg"]["channels"])}
bip = S["ieeg"]["data"][idx, c["AHL2"]].astype(float) - S["ieeg"]["data"][idx, c["AHL3"]].astype(float)
E = wmflow.ecog_signals(S, idx)
rng = np.random.default_rng(32)
for per in ["enc", "maint"]:
    hx, cx = wmflow.window40(bip, per), wmflow.window40(E["GL_C2"], per)
    r = wmflow.granger(hx, cx)
    null = np.stack([wmflow.granger(hx, cx[rng.permutation(len(cx))])["dg"] for _ in range(200)])
    sel = (r["freq"] >= 4) & (r["freq"] <= 20)
    lo, hi = np.percentile(null, [1, 99], axis=0)
    print("code variant", per, "| d1 %.4f" % wmflow.band_value(r["freq"], r["dg"]),
          "| below 1st percentile:", wmflow.runs((r["dg"] < lo)[sel], r["freq"][sel]),
          "| above 99th:", wmflow.runs((r["dg"] > hi)[sel], r["freq"][sel]))

**Result: reproduced, resting on reading h1 and the per-frequency test (c1).** With h1 (AHL1
against a white-matter contact), ΔGranger between the hippocampus and C2 is below the null's 5th
percentile in encoding at 6.75 to 7.2 and 7.55 to 9.05 Hz (theta mean -0.012) and above its 95th
percentile in maintenance at 7.0 to 7.45 Hz (theta mean -0.0003). The other readings do not show
both: h2 is significant in maintenance only (6.15 to 7.9 Hz), h3 in neither; with clusters (c2)
the maintenance band is not significant under any reading. The code's variant (bipolar AHL2-AHL3,
trial-shuffle null, 1st and 99th percentiles) finds encoding at 5.9 to 6.15 Hz and nothing in
maintenance. The direction-swap null is narrow: the same test finds significant frequencies in
all 24 tests of h1 in Participant 4 (fig3no-p4-right-no-gc), so a significant band says little.

## fig2ijk-spatial-spread-p1

> In Participant 1, the strongest net flow in theta during encoding was from auditory cortex
> contacts to hippocampus; during maintenance theta flow from hippocampus was significant to
> auditory and occipital cortex, and this spatial pattern was significant against a null
> distribution.

ΔGranger over [4 8] Hz per grid contact (d1, d2 shown). Per contact, the direction test of
fig2h-gc-reversal-p1 on the band value, for reading h1 (h2 and h3 are shown without the test,
to keep the notebook within its time limit). The spatial test runs on maintenance with h1
(200 iterations); its other readings were not run, for time.

In [ ]:
t0 = time.time()
S = ALL["sub-10"]; idx = trial_idx(S)
H1 = hipp_reading_signals(S, idx)
E = wmflow.ecog_signals(S, idx)
rows = []
for h in ["h1", "h2", "h3"]:
    for per in ["enc", "maint"]:
        for c in GRID:
            cx = wmflow.window40(E[c], per)
            if h != "h1":
                r = wmflow.gc_reading(H1[(h, per)], cx)
                p_band = np.nan
                d1 = wmflow.band_value(r["freq"], r["dg"]); d2 = wmflow.band_value(r["freq"], r["dg"], how="ext", period=per)
            else:
                g = wmflow.gc_test(H1[(h, per)], cx, 200, 41)
                th = (g["freq"] >= 4) & (g["freq"] <= 8)
                d1 = g["dg"][th].mean(); d2 = wmflow.band_value(g["freq"], g["dg"], how="ext", period=per)
                nb = g["null"][:, th].mean(1)
                p_band = (1 + ((nb <= d1) if per == "enc" else (nb >= d1)).sum()) / 201
            rows.append(dict(h=h, period=per, contact=c, region=REGION[c], auditory=c in AUDITORY, d1=d1, d2=d2, p=p_band))
grid_gc = pd.DataFrame(rows)
print("%.0f s" % (time.time() - t0))
for (h, per), d in grid_gc.groupby(["h", "period"], sort=False):
    d = d.set_index("contact")
    ext = d.d1.idxmin() if per == "enc" else d.d1.idxmax()
    ext2 = d.d2.idxmin() if per == "enc" else d.d2.idxmax()
    sig = d[d.p < 0.05].index.tolist()
    print(f"{h} {per}: d1 range {d.d1.min():.4f} to {d.d1.max():.4f}; most {'negative' if per == 'enc' else 'positive'} d1 at {ext} "
          f"({'auditory' if ext in AUDITORY else REGION[ext]}), d2 at {ext2} ({'auditory' if ext2 in AUDITORY else REGION[ext2]}); "
          f"significant contacts {len(sig)}: auditory {[x for x in sig if x in AUDITORY]}, occipital H3 {'GL_H3' in sig}")
print("ΔGranger d1 [4 8] Hz, maintenance, h1:")
grid_print(grid_gc[(grid_gc.h == "h1") & (grid_gc.period == "maint")].set_index("contact").d1, "{:8.4f}")
print("ΔGranger d1 [4 8] Hz, encoding, h1:")
grid_print(grid_gc[(grid_gc.h == "h1") & (grid_gc.period == "enc")].set_index("contact").d1, "{:8.4f}")

In [ ]:
t0 = time.time()
hx_m = H1[("h1", "maint")][0]
t40 = -6.0 + np.arange(320) / 40
hs_full = wmflow.hipp_signals(S["ieeg"]["data"][idx].astype(float), S["ieeg"]["channels"], S["hc"])
hx_f = wmflow.cut(wmflow.to_fs(hs_full["h1"], 500), t40, wmflow.WINDOWS["fix"], 40)
csd_task = {c: wmflow.trial_csds(hx_m, wmflow.window40(E[c], "maint")) for c in GRID}
csd_fix = {c: wmflow.trial_csds(hx_f, wmflow.window40(E[c], "fix")) for c in GRID}
sp = wmflow.spatial_test(csd_task, csd_fix, 200, np.random.default_rng(42))
q95 = np.percentile(sp["null"], 95)
print("%.0f s" % (time.time() - t0))
print("scalar product, true: median %.3g (5-95%%: %.3g to %.3g); null: median %.3g, 95th percentile %.3g" % (
    np.median(sp["true"]), *np.percentile(sp["true"], [5, 95]), np.median(sp["null"]), q95))
print("s1 (median of true above the null 95th percentile):", bool(np.median(sp["true"]) > q95),
      "| s2 (at least 95% of true values above it):", bool((sp["true"] > q95).mean() >= 0.95),
      "| fraction above: %.2f" % (sp["true"] > q95).mean())

**Result: not reproduced.** In encoding the most negative theta ΔGranger (h1) is at G6, an
occipital contact, not an auditory one (h2: G6; h3: H5). Of the 7 contacts with significant
positive ΔGranger in maintenance (h1), none is auditory and H3 is not among them. The spatial
test is not significant: the scalar products of the two halves' grid vectors have a median of
-0.0041, below the null's median (-0.0012) and its 95th percentile (0.013).

## fig3-gc-reversal-p2-p3

> The same reversal appeared in Participant 2 between auditory cortex and hippocampus (encoding
> [6 8] Hz, maintenance [6 10] Hz) and in Participant 3 between visual cortex and the right
> hippocampus (encoding 6 to 9 Hz, maintenance [6 8] Hz).

Participant 2: the most anterior strip contact by MNI y. Participant 3: the most posterior strip
contact, with the hippocampal readings taken from the right hippocampal electrodes (AHR, PHR), as
the criterion requires.

In [ ]:
ec2 = ALL["sub-11"]["electrodes"]; ec2 = ec2[ec2.ecog].sort_values("y")
ec3 = ALL["sub-12"]["electrodes"]; ec3 = ec3[ec3.ecog].sort_values("y")
c_p2, c_p3 = ec2.label.iloc[-1], ec3.label.iloc[0]
print("Participant 2 most anterior contact", c_p2, "y =", ec2.y.iloc[-1], "| Participant 3 most posterior contact", c_p3, "y =", ec3.y.iloc[0])
hc3 = dict(ALL["sub-12"]["hc"])
hc3.update(h1="AHR1", h2=("AHR1", "AHR2"), h3=["AHR1", "PHR1"])
print("Participant 3 right-hippocampus readings:", hc3["h1"], hc3["h2"], hc3["h3"], "ref", hc3["ref"])
t0 = time.time()
gc_p2, spec_p2 = ecog_gc_rows("sub-11", [c_p2], seed=51)
gc_p3, spec_p3 = ecog_gc_rows("sub-12", [c_p3], seed=52, hc=hc3)
gc_p3l, _ = ecog_gc_rows("sub-12", [c_p3], seed=53)
gc_p3l["h"] = gc_p3l.h + " (left)"
print("%.0f s" % (time.time() - t0))
gc23 = pd.concat([gc_p2, gc_p3, gc_p3l])
print(gc23.to_string())
for (sub, h), d in gc23.groupby(["sub", "h"], sort=False):
    d = d.set_index("period")
    print(sub, h, "| c1 reversal in theta:", bool(d.loc["enc", "c1_theta"] and d.loc["maint", "c1_theta"]),
          "| c2:", bool(d.loc["enc", "c2_theta"] and d.loc["maint", "c2_theta"]))

**Result: partially reproduced, resting on reading h2 and the per-frequency test.** In
Participant 3 (right hippocampus), h2 shows the reversal with the posterior contact PL1:
significantly negative ΔGranger in encoding at 6.65 to 7.55 Hz and positive in maintenance at
4.2 to 4.95 and 6.75 to 7.6 Hz; h1 and h3 are significant in maintenance only (4 to 10 Hz), and
with the left hippocampus no reading shows it. In Participant 2 no reading shows the reversal
with the most anterior contact TBAL1 (h2: maintenance 4.0 to 6.75 Hz only). The claim holds for
one of the two participants.

## fig3no-p4-right-no-gc

> In Participant 4, recordings from right parietal cortex showed no significant GC with the
> hippocampus during the task.

A claim of absence, judged on the paper's test (direction-swap null, 200 permutations, p < 0.05
per frequency over [4 20] Hz) on every ECoG contact of Participant 4 (all on the right).

In [ ]:
t0 = time.time()
ec4 = ALL["sub-13"]["electrodes"]; ec4 = ec4[ec4.ecog]
gc_p4, spec_p4 = ecog_gc_rows("sub-13", list(ec4.label), seed=61)
print("%.0f s" % (time.time() - t0))
gc_p4["region"] = gc_p4.contact.map(ec4.set_index("label").AnatomicalLocation.str.split(",").str[0])
gc_p4["any_sig_4_20"] = gc_p4.c1_bands.map(len) > 0
gc_p4["sig_theta_only"] = gc_p4.c1_theta
print(gc_p4[["contact", "region", "period", "h", "d1", "d2", "c1_bands", "c2_clusters"]].to_string())
summ = gc_p4.groupby("h").agg(contacts=("contact", "nunique"), tests=("contact", "size"),
                              with_sig_4_20=("any_sig_4_20", "sum"), with_sig_theta=("sig_theta_only", "sum"),
                              with_c2_cluster=("c2_clusters", lambda x: int(sum(len(v) > 0 for v in x))),
                              max_abs_d1=("d1", lambda x: np.abs(x).max()))
print(summ.to_string())
# two-sided variant: either direction beyond the 2.5th or 97.5th percentile
two = []
for (c, per, h), g in spec_p4.items():
    sel = (g["freq"] >= 4) & (g["freq"] <= 20)
    lo, hi = np.percentile(g["null"][:, sel], [2.5, 97.5], axis=0)
    two.append(bool(((g["dg"][sel] < lo) | (g["dg"][sel] > hi)).any()))
print("two-sided variant: contact-period-reading tests with any significant frequency:", sum(two), "of", len(two))

**Result: not reproduced.** The paper's test finds significant ΔGranger at some frequency of
[4 20] Hz in 24 of 24 contact and period tests with h1, 22 of 24 with h2 and 23 of 24 with h3,
and within [4 8] Hz in 20, 16 and 18; the largest theta ΔGranger is 0.043 (h1), 0.064 (h2) and
0.018 (h3). The direction-swap null is narrow enough that almost any contact passes it, in
this participant as in Participant 1.

## table1-left-hemisphere-sources

> In each participant, more than 80% of the EEG sources that differed significantly from fixation
> lay in the left hemisphere, in encoding and in maintenance.

Blocked, `not_runnable`. The test needs LCMV beamforming of the scalp EEG on an AAL grid and
FieldTrip's cluster-based source statistics. The paper's code aligns each participant's
electrodes to a template head model by hand (`ft_electroderealign` with `cfg.method =
'interactive'` in MainSourceAnalysisConnectivity.m), a step that cannot be repeated, and FieldTrip
runs in MATLAB, which cannot be installed in the entry's locked environment. The dataset holds
five averaged virtual sensors per participant, not the source grid, so the hemisphere shares
cannot be read from it either.

## The hippocampus and the EEG sources (Participants 1 to 15)

The derivative files hold, per participant, the virtual sensors AC, DLPFC, OFC, PPC and V1 for
the GC method's trials (matched to the events above). The claims file's readings of the auditory
source are (a1) the AC virtual sensor, (a2) left Heschl gyrus and (a3) left superior temporal
gyrus; only a1 is in the dataset, so a2 and a3 cannot be run. GC per participant for every area,
hippocampal reading (h1 to h3) and trial reading: (t1) all of the GC method's trials, and (t2)
10% of them drawn at random, 200 draws for AC (Methods, Statistics, paragraph 4) and 20 draws for
the other areas (the code's `nIterations = 20`, to keep the run short), aggregated by the median
(Results, code) or the mean (Methods) of the direction spectra over draws. ΔGranger over [4 8] Hz
is the mean (d1) or the extreme (d2: minimum in encoding, maximum in maintenance).

In [ ]:
t0 = time.time()
ELIFE = part.set_index("participant_id").eLife_identifier.astype(int).to_dict()
SUBS = sorted(ALL)
G1, G2 = {}, {}
rng_t2 = np.random.default_rng(71)
for sub in SUBS:
    S = ALL[sub]
    keep, ii = wmflow.matched_trials(S)
    H = wmflow.hipp40(S, ii)
    for area in wmflow.AREAS:
        for per in ["enc", "maint"]:
            cx = wmflow.source40(S, keep, area, per)
            for h in ["h1", "h2", "h3"]:
                G1[(sub, area, per, h)] = wmflow.gc_reading(H[(h, per)], cx)
                G2[(sub, area, per, h)] = wmflow.gc_subsample(H[(h, per)], cx, 0.10, 200 if area == "AC" else 20, rng_t2)
FREQ = G1[(SUBS[0], "AC", "enc", "h1")]["freq"]
print("GC computed in %.0f s; trials per participant:" % (time.time() - t0),
      {s: int(len(wmflow.matched_trials(ALL[s])[0])) for s in SUBS}, "; 10% draws of",
      {s: G2[(s, "AC", "enc", "h1")]["k"] for s in SUBS}, "trials")

def dg_value(sub, area, per, h, t="t1", d="d1", agg="median", flip_maint=False):
    """Per-participant ΔGranger [4 8] Hz for one reading. ``flip_maint`` is the code's labelling."""
    if t == "t1":
        r = G1[(sub, area, per, h)]; hc, ch = r["hc"], r["ch"]
    else:
        r = G2[(sub, area, per, h)]; hc, ch = getattr(np, agg)(r["hc"], 0), getattr(np, agg)(r["ch"], 0)
    dg = hc - ch
    if flip_maint and per == "maint":
        dg = -dg
    return wmflow.band_value(FREQ, dg, how="mean" if d == "d1" else "ext", period=per)

READINGS = [(h, t, agg, d) for h in ["h1", "h2", "h3"] for t, agg in [("t1", None), ("t2", "median"), ("t2", "mean")] for d in ["d1", "d2"]]
def rname(r):
    h, t, agg, d = r
    return f"{h}-{t}{'-' + agg if agg else ''}-{d}"
def values(area, reading, **kw):
    h, t, agg, d = reading
    return pd.DataFrame({per: [dg_value(s, area, per, h, t, d, agg or "median", **kw) for s in SUBS] for per in ["enc", "maint"]}, index=SUBS)

## fig4a-group-gc-reversal

> Across the 15 participants, GC between hippocampal LFP and auditory cortex EEG sources showed net
> flow from auditory cortex to hippocampus during encoding ([6 10] Hz) and from hippocampus to
> auditory cortex during maintenance ([6 9] Hz and [13 15] Hz).

Per frequency over [4 20] Hz, the two directions' spectra of the 15 participants are compared by a
Monte Carlo t-test with 2500 randomisations and FDR correction (independent samples, as the
code; dependent samples as the variant). All of the GC method's trials (t1).

In [ ]:
rng = np.random.default_rng(81)
sel = (FREQ >= 4) & (FREQ <= 20)
rows = []
for h in ["h1", "h2", "h3"]:
    for paired in [False, True]:
        res = {}
        for per in ["enc", "maint"]:
            hc = np.stack([G1[(s, "AC", per, h)]["hc"][sel] for s in SUBS])
            ch = np.stack([G1[(s, "AC", per, h)]["ch"][sel] for s in SUBS])
            st = wmflow.group_spectra_test(hc, ch, 2500, rng, paired=paired)
            res[per] = st
            neg = wmflow.runs(st["sig"] & (st["t"] < 0), FREQ[sel])  # cortex -> hipp larger
            pos = wmflow.runs(st["sig"] & (st["t"] > 0), FREQ[sel])  # hipp -> cortex larger
            rows.append(dict(h=h, test="dependent" if paired else "independent", period=per,
                             ctx_to_hipp_larger=neg, hipp_to_ctx_larger=pos, min_p=st["p"].min(),
                             mean_dg_theta=(hc - ch)[:, (FREQ[sel] >= 4) & (FREQ[sel] <= 8)].mean()))
g4a = pd.DataFrame(rows)
print(g4a.to_string())
ok = []
for (h, test), d in g4a.groupby(["h", "test"]):
    d = d.set_index("period")
    e_ok = any(a <= 10 and b >= 6 for a, b in d.loc["enc", "ctx_to_hipp_larger"])
    m_ok = any(a <= 9 and b >= 6 for a, b in d.loc["maint", "hipp_to_ctx_larger"])
    ok.append((h, test, e_ok, m_ok))
    print(h, test, "| encoding cortex->hipp overlapping [6 10] Hz:", e_ok, "| maintenance hipp->cortex overlapping [6 9] Hz:", m_ok)
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for a, per in zip(ax, ["enc", "maint"]):
    for key, col in [("hc", "tab:red"), ("ch", "tab:blue")]:
        m = np.mean([G1[(s, "AC", per, "h1")][key][sel] for s in SUBS], 0)
        a.plot(FREQ[sel], m, color=col, label={"hc": "hipp→AC", "ch": "AC→hipp"}[key])
    a.set_title(f"{per}, h1, mean of 15"); a.set_xlabel("Hz")
ax[0].set_ylabel("GC"); ax[0].legend()
allm = [np.mean([G1[(s, "AC", per, "h1")][k][sel] for s in SUBS], 0) for per in ["enc", "maint"] for k in ["hc", "ch"]]
print("plotted mean GC range %.4f to %.4f" % (np.min(allm), np.max(allm)))
plt.show()

**Result: not reproduced.** No frequency in [4 20] Hz differs significantly between the two
directions after FDR correction, in either period, under any hippocampal reading or either
t-test (smallest uncorrected Monte Carlo p 0.004 to 0.030). The group-mean theta ΔGranger is
positive in both periods (0.0015 to 0.0058).

## scalp-temporal-laplacian

> A similar GC pattern appeared with the signals of left temporal scalp electrodes, and it
> disappeared with a Laplacian derivation.

The derivative files carry the scalp channels for the same trials. Their reference is checked
first (if they are referenced to the averaged mastoids, A1 = -A2). Electrode readings (e1) T3,
(e2) T5, (e3) the mean of the two GC spectra; Laplacian by spherical splines (MNE's
`compute_current_source_density`, standing in for FieldTrip's `ft_scalpcurrentdensity`, on the
standard 10-20 positions), with the Hjorth derivation (the electrode minus the mean of its
neighbours) as the variant. Tests (s1) the paired permutation test of encoding against
maintenance ΔGranger (200 sign flips) and (s2) the group spectra test of fig4a. Only participants
whose derivative files hold the electrode enter.

In [ ]:
import mne
mont = mne.channels.make_standard_montage("standard_1020")
HJORTH = {"T3": ["F7", "C3", "T5"], "T5": ["T3", "P3", "O1"]}
ref_chk = []
for s in SUBS:
    L = ALL[s]["lcmv"]["enc"]
    if "A1" in L["label"] and "A2" in L["label"]:
        a1, a2 = L["data"][:, L["label"].index("A1")], L["data"][:, L["label"].index("A2")]
        ref_chk.append(dict(sub=s, corr_A1_A2=np.corrcoef(a1.ravel(), a2.ravel())[0, 1], rms_sum_over_rms=np.std(a1 + a2) / np.std(a1)))
print(pd.DataFrame(ref_chk).round(3).to_string())

def scalp_signals(S, keep, per, deriv):
    L = S["lcmv"][per]
    labs = L["label"][5:]
    X = L["data"][keep][:, 5:]
    out = {}
    for e in ["T3", "T5"]:
        if e not in labs:
            continue
        x = X[:, labs.index(e)]
        if deriv == "mastoid":
            out[e] = x
        elif deriv == "hjorth":
            nb = [n for n in HJORTH[e] if n in labs]
            if len(nb) < 2:
                continue
            out[e] = x - X[:, [labs.index(n) for n in nb]].mean(1)
        else:
            eeg = [l for l in labs if l in mont.ch_names and l not in ("A1", "A2")]
            info = mne.create_info(eeg, L["fs"], "eeg")
            ep = mne.EpochsArray(X[:, [labs.index(l) for l in eeg]] * 1e-6, info, verbose=False)
            ep.set_montage(mont, verbose=False)
            ep = mne.preprocessing.compute_current_source_density(ep, verbose=False)
            out[e] = ep.get_data()[:, eeg.index(e)]
    return {e: wmflow.to_fs(v, L["fs"])[:, :80] for e, v in out.items()}

rows, spec_sc = [], {}
for s in SUBS:
    S = ALL[s]
    keep, ii = wmflow.matched_trials(S)
    H = wmflow.hipp40(S, ii)
    for deriv in ["mastoid", "spline", "hjorth"]:
        sig = {per: scalp_signals(S, keep, per, deriv) for per in ["enc", "maint"]}
        for h in ["h1", "h2", "h3"]:
            for e in ["e1", "e2", "e3"]:
                els = {"e1": ["T3"], "e2": ["T5"], "e3": ["T3", "T5"]}[e]
                if not all(x in sig["enc"] for x in els):
                    continue
                for per in ["enc", "maint"]:
                    gs = [wmflow.gc_reading(H[(h, per)], sig[per][x]) for x in els]
                    hc = np.mean([g["hc"] for g in gs], 0); ch = np.mean([g["ch"] for g in gs], 0)
                    spec_sc[(s, deriv, h, e, per)] = (hc, ch)
                    rows.append(dict(sub=s, deriv=deriv, h=h, e=e, period=per,
                                     d1=wmflow.band_value(FREQ, hc - ch), d2=wmflow.band_value(FREQ, hc - ch, how="ext", period=per)))
sc = pd.DataFrame(rows)
rng = np.random.default_rng(82)
out = []
for (deriv, h, e), d in sc.groupby(["deriv", "h", "e"]):
    subs_e = sorted(d["sub"].unique())
    for dd in ["d1", "d2"]:
        w = d.pivot(index="sub", columns="period", values=dd)
        p1 = wmflow.paired_signflip(w.maint.values, w.enc.values, 200, rng)
        hcs = {per: np.stack([spec_sc[(s, deriv, h, e, per)][0][sel] for s in subs_e]) for per in ["enc", "maint"]}
        chs = {per: np.stack([spec_sc[(s, deriv, h, e, per)][1][sel] for s in subs_e]) for per in ["enc", "maint"]}
        st = {per: wmflow.group_spectra_test(hcs[per], chs[per], 500, rng) for per in ["enc", "maint"]}
        s2 = (any(a <= 8 and b >= 4 for a, b in wmflow.runs(st["enc"]["sig"] & (st["enc"]["t"] < 0), FREQ[sel]))
              and any(a <= 8 and b >= 4 for a, b in wmflow.runs(st["maint"]["sig"] & (st["maint"]["t"] > 0), FREQ[sel])))
        out.append(dict(deriv=deriv, h=h, e=e, d=dd, n=len(w), enc_median=w.enc.median(), maint_median=w.maint.median(),
                        s1_p=p1, s1_enc_below_maint=w.enc.mean() < w.maint.mean(), s2_reversal_in_theta=s2))
sct = pd.DataFrame(out)
print(sct.round(4).to_string())
piv = sct.assign(s1=(sct.s1_p < 0.05) & sct.s1_enc_below_maint).pivot_table(index=["h", "e", "d"], columns="deriv", values=["s1", "s2_reversal_in_theta"], aggfunc="first")
print(piv.to_string())
for test in ["s1", "s2_reversal_in_theta"]:
    hit = piv[test]
    print(test, "| significant with the mastoid reference and not with the spline Laplacian:", int((hit.mastoid & ~hit.spline).sum()),
          "of", len(hit), "| not with Hjorth:", int((hit.mastoid & ~hit.hjorth).sum()))
print("participants with T3:", sorted(sc[sc.e == "e1"]["sub"].unique()), "| with T5:", sorted(sc[sc.e == "e2"]["sub"].unique()))

**Result: not reproduced.** The derivative scalp channels are referenced to the averaged
mastoids (A1 = -A2 in every participant). With the mastoid reference, the paired test finds
encoding below maintenance under every d2 reading (18 of 18, p = 0.005 to 0.015) and under no d1
reading; the group test finds no reversal under any reading. After the spline Laplacian the
same d2 tests remain significant (18 of 18), and after the Hjorth derivation too, so the
pattern is not eliminated by the Laplacian under any reading. 8 participants have T3 and 10
have T5 in the derivative files.

## fig4bc-auditory-strongest

> Theta net flow was strongest between hippocampus and auditory cortex, toward the hippocampus
> during encoding (ΔGranger -0.049, p = 0.0009) and toward the cortex during maintenance
> (ΔGranger 0.034, p = 0.001); other areas showed weaker flow in the same directions.

Per reading (h, t, d) and group statistic (g1 mean, g2 median): the group value of each area,
a Kruskal-Wallis test across the five areas' participant values per period, and Dunn's test
with Bonferroni correction.

In [ ]:
rows = []
VAL = {}
for rd in READINGS:
    V = {a: values(a, rd) for a in wmflow.AREAS}
    VAL[rd] = V
    for per in ["enc", "maint"]:
        kw = stats.kruskal(*[V[a][per].values for a in wmflow.AREAS])
        for g in ["mean", "median"]:
            grp = {a: getattr(np, g)(V[a][per].values) for a in wmflow.AREAS}
            best = min(grp, key=grp.get) if per == "enc" else max(grp, key=grp.get)
            rows.append(dict(reading=rname(rd), period=per, g=g, AC=grp["AC"], DLPFC=grp["DLPFC"], OFC=grp["OFC"], PPC=grp["PPC"],
                             V1=grp["V1"], kw_p=kw.pvalue, extreme_area=best))
bc = pd.DataFrame(rows)
print(bc.round(4).to_string())
bc["ac_in_tol"] = np.where(bc.period == "enc", (bc.AC + 0.049).abs() <= 0.0049, (bc.AC - 0.034).abs() <= 0.0034)
w = bc.pivot_table(index=["reading", "g"], columns="period", values=["ac_in_tol", "kw_p", "extreme_area"], aggfunc="first")
w[("all_hold", "")] = w[("ac_in_tol", "enc")] & w[("ac_in_tol", "maint")] & (w[("kw_p", "enc")] < 0.05) & (w[("kw_p", "maint")] < 0.05) \
    & (w[("extreme_area", "enc")] == "AC") & (w[("extreme_area", "maint")] == "AC")
print("readings where every part of the criterion holds:", int(w[("all_hold", "")].sum()), "of", len(w))
print("readings with AC within tolerance in both periods:", int((w[("ac_in_tol", "enc")] & w[("ac_in_tol", "maint")]).sum()),
      "| with Kruskal-Wallis p < 0.05 in both periods:", int(((w[("kw_p", "enc")] < 0.05) & (w[("kw_p", "maint")] < 0.05)).sum()),
      "| with AC the extreme area in both:", int(((w[("extreme_area", "enc")] == "AC") & (w[("extreme_area", "maint")] == "AC")).sum()))
for rd in [("h1", "t2", "mean", "d2"), ("h1", "t1", None, "d1")]:
    for per in ["enc", "maint"]:
        print(rname(rd), per, "Dunn-Bonferroni pairs with AC:")
        dn = wmflow.dunn_bonferroni({a: VAL[rd][a][per].values for a in wmflow.AREAS})
        print(dn[(dn.a == "AC") | (dn.b == "AC")].round(4).to_string(index=False))

**Result: not reproduced.** Under no reading do all parts hold. One reading (h1, 10% draws, mean
over draws, d2, group mean) puts the auditory value within tolerance in both periods (-0.045 and
0.035), but under it the Kruskal-Wallis test is not significant (p = 0.33 and 0.44) and other
areas are more extreme (DLPFC in encoding, OFC in maintenance). The Kruskal-Wallis test reaches
p < 0.05 in one period of one reading only (h2-t2-median-d2, encoding, p = 0.009, with DLPFC the
most negative), and the auditory source is never the most extreme area in both periods. Dunn's
tests find no pair with AC significant.

## fig4d-reversal-all

> The reversal of theta net flow between hippocampus and auditory cortex appeared in each of the
> 15 participants, negative in encoding (median -0.049) and positive in maintenance (median
> 0.034), p = 4.1e-10 by a paired permutation test, correct trials only.

Per reading: the sign pattern per participant, the medians, the paired permutation test of
encoding against maintenance (200 sign flips, Methods, Statistics, paragraph 5; the smallest p it
can give is 1/201 = 0.005) and the exact sign-flip p over all 2^15 flips. Two controls follow:
the same statistic computed on the direction-swap null (what d2 gives when there is no
direction at all), and the code's labelling of the directions.

In [ ]:
rng = np.random.default_rng(91)
rows = []
for rd in READINGS:
    v = VAL[rd]["AC"]
    rows.append(dict(reading=rname(rd), enc_neg=int((v.enc < 0).sum()), maint_pos=int((v.maint > 0).sum()),
                     both=int(((v.enc < 0) & (v.maint > 0)).sum()), enc_median=v.enc.median(), maint_median=v.maint.median(),
                     p200=wmflow.paired_signflip(v.maint.values, v.enc.values, 200, rng),
                     p_exact=wmflow.paired_signflip(v.maint.values, v.enc.values, None)))
d4 = pd.DataFrame(rows)
d4["medians_in_tol"] = ((d4.enc_median + 0.049).abs() <= 0.0049) & ((d4.maint_median - 0.034).abs() <= 0.0034)
d4["reproduced"] = (d4.both == 15) & d4.medians_in_tol & (d4.p200 < 0.05)
d4["partial"] = d4.both.between(12, 14) & d4.medians_in_tol & (d4.p200 < 0.05)
print(d4.round(4).to_string())
print("readings meeting the criterion:", int(d4.reproduced.sum()), "| partially:", int(d4.partial.sum()),
      "| with all 15 participants in the pattern:", int((d4.both == 15).sum()))
print("\nper participant (eLife number), reading h1-t2-mean-d2:")
v = VAL[("h1", "t2", "mean", "d2")]["AC"].copy(); v["eLife"] = [ELIFE[s] for s in v.index]
print(v.sort_values("eLife").round(4).to_string())

In [ ]:
# Control 1: the d2 statistic on spectra with no direction. For each participant the direction-swap
# null (20 draws) gives ΔGranger spectra whose expected value is 0; d2 is taken from each draw.
rng = np.random.default_rng(92)
ctrl = []
for s in SUBS:
    S = ALL[s]; keep, ii = wmflow.matched_trials(S); H = wmflow.hipp40(S, ii)
    for per in ["enc", "maint"]:
        cx = wmflow.source40(S, keep, "AC", per)
        nl, f = wmflow.granger_direction_null(H[("h1", per)][0], cx, 20, rng)
        ctrl.append(dict(sub=s, period=per, null_d2_median=np.median(wmflow.band_value(f, nl, how="ext", period=per)),
                         null_d1_median=np.median(wmflow.band_value(f, nl))))
ctrl = pd.DataFrame(ctrl).pivot(index="sub", columns="period")
print(ctrl.round(4).to_string())
print("d2 on the no-direction null, h1 t1: encoding negative in", int((ctrl[("null_d2_median", "enc")] < 0).sum()),
      "and maintenance positive in", int((ctrl[("null_d2_median", "maint")] > 0).sum()), "of 15; medians %.4f and %.4f" % (
      ctrl[("null_d2_median", "enc")].median(), ctrl[("null_d2_median", "maint")].median()))
# Control 2: the opposite extreme. If d2's sign pattern came from the task, taking the maximum in
# encoding and the minimum in maintenance would not give the mirror pattern.
for h in ["h1", "h2", "h3"]:
    e_max = [wmflow.band_value(FREQ, G1[(s, "AC", "enc", h)]["dg"], how="ext", period="maint") for s in SUBS]
    m_min = [wmflow.band_value(FREQ, G1[(s, "AC", "maint", h)]["dg"], how="ext", period="enc") for s in SUBS]
    v = VAL[(h, "t1", None, "d2")]["AC"]
    print(f"{h} t1 | d2 as defined: encoding min < 0 in {int((v.enc < 0).sum())}, maintenance max > 0 in {int((v.maint > 0).sum())} | "
          f"opposite extremes: encoding max > 0 in {sum(x > 0 for x in e_max)}, maintenance min < 0 in {sum(x < 0 for x in m_min)}; "
          f"medians {np.median(e_max):.4f} and {np.median(m_min):.4f}; paired p200 (max enc vs min maint) "
          f"{wmflow.paired_signflip(np.array(e_max), np.array(m_min), 200, np.random.default_rng(94)):.3f}")
# Control 3: the code's labelling (calculateGranger_random_trial_sampling.m stores grangerspctrm row 2 as
# cortex->hipp in encoding and as hipp->cortex in maintenance), which flips the sign of maintenance.
for rd in [("h1", "t2", "median", "d2"), ("h1", "t2", "median", "d1"), ("h1", "t1", None, "d1")]:
    v = values("AC", rd, flip_maint=True)
    print("code labelling,", rname(rd), "| encoding negative", int((v.enc < 0).sum()), "| maintenance positive", int((v.maint > 0).sum()),
          "| medians %.4f %.4f" % (v.enc.median(), v.maint.median()))

In [ ]:
# Variant: the three ECoG values of Figure 4d's code (P1 C2, P2 anterior strip, P3 posterior strip with the
# right hippocampus) pooled with the 15 source values, reading h1-t1-d2.
ec = {("sub-10", "enc"): gc_p1, ("sub-11", "enc"): gc_p2, ("sub-12", "enc"): gc_p3}
pool = {per: list(VAL[("h1", "t1", None, "d2")]["AC"][per].values) for per in ["enc", "maint"]}
for df in [gc_p1, gc_p2, gc_p3]:
    for per in ["enc", "maint"]:
        pool[per].append(float(df[(df.h == "h1") & (df.period == per)].d2.iloc[0]))
print("pooled 18 values, h1-t1-d2: medians %.4f (enc) and %.4f (maint); pattern in %d of 18; p200 = %.3f" % (
    np.median(pool["enc"]), np.median(pool["maint"]), int(sum(e < 0 and m > 0 for e, m in zip(pool["enc"], pool["maint"]))),
    wmflow.paired_signflip(np.array(pool["maint"]), np.array(pool["enc"]), 200, np.random.default_rng(93))))

**Result: not reproduced.** No reading meets the criterion, because the medians are about half
the paper's: the closest is h1 with 10% draws, mean over draws and d2, at -0.035 (encoding) and
0.019 (maintenance) against -0.049 and 0.034. With the extreme value d2 the sign pattern holds in
13 to 15 participants and the paired test gives p = 0.005 to 0.010 with 200 permutations (0.005
is the smallest it can give; exact sign-flip p = 0.0001). With the band mean d1, which the Results' "We averaged ΔGranger"
describes, there is no effect: 1 to 5 participants show the pattern, medians are within 0.004
of zero and p = 0.28 to 0.86 (200 permutations). The controls show that d2's sign pattern comes from the statistic:
the opposite extremes (maximum in encoding, minimum in maintenance) give the mirror pattern
(printed above), and the d2 of a spectrum with no direction at all is negative in encoding and
positive in maintenance in 15 of 15 participants.

## table1-heschl-dgranger-correct

> Table 1 reports, for each participant, a negative ΔGranger to Heschl gyrus in encoding and a
> positive one in maintenance on correct trials.

The paper's values (Table 1, as the claims file gives them) against this attempt's, matched by
the `eLife_identifier` column of participants.tsv. Tolerance 10% of each value and at least 0.001.

In [ ]:
T1_ENC = [-0.036, -0.017, -0.060, -0.006, -0.003, -0.049, -0.059, -0.040, -0.051, -0.070, -0.052, -0.022, -0.018, -0.053, -0.061]
T1_MAINT = [0.025, 0.093, 0.065, 0.035, 0.021, 0.039, 0.042, 0.017, 0.050, 0.018, 0.020, 0.012, 0.020, 0.085, 0.069]
paper = pd.DataFrame({"enc": T1_ENC, "maint": T1_MAINT}, index=range(1, 16))
pathology_t1 = {5: "brain contusion", 6: "hippocampal sclerosis", 15: "hippocampal sclerosis"}
print("pathology in participants.tsv against Table 1 where they differ:",
      {k: (part.set_index("eLife_identifier").loc[k, "pathology"], v) for k, v in pathology_t1.items()})
rows = []
for rd in READINGS:
    v = VAL[rd]["AC"].copy(); v.index = [ELIFE[s] for s in v.index]; v = v.sort_index()
    tol = np.maximum(0.1 * paper.abs(), 0.001)
    sign = (np.sign(v) == np.sign(paper)).values.sum()
    within = ((v - paper).abs() <= tol).values.sum()
    rows.append(dict(reading=rname(rd), signs_match=int(sign), within_tol=int(within),
                     corr_enc=np.corrcoef(v.enc, paper.enc)[0, 1], corr_maint=np.corrcoef(v.maint, paper.maint)[0, 1]))
t1 = pd.DataFrame(rows)
print(t1.round(3).to_string())
print("readings with all 30 signs:", int((t1.signs_match == 30).sum()), "| with at least 26:", int((t1.signs_match >= 26).sum()),
      "| most cells within tolerance:", int(t1.within_tol.max()))
best = max(READINGS, key=lambda r: (int(t1.set_index("reading").loc[rname(r), "signs_match"]), int(t1.set_index("reading").loc[rname(r), "within_tol"])))
v = VAL[best]["AC"].copy(); v.index = [ELIFE[s] for s in v.index]
print("\nreading", rname(best), "against Table 1:")
print(pd.concat([v.sort_index().add_prefix("here_"), paper.add_prefix("paper_")], axis=1).round(3).to_string())

**Result: partially reproduced, resting on reading d2.** Under five d2 readings (h2 and h3,
all trials or 10% draws) the signs match Table 1 in all 30 cells, and under four more in 28 or
29; but at most 3 of 30 values are within tolerance, short of the half the criterion asks, and
the values do not correlate with Table 1 (r = -0.51 to 0.38 over the readings). Under d1 the signs match in 13 to
19 cells. As fig4d-reversal-all shows, d2 makes this sign pattern without a directed effect.

## fig4e-lpfc-no-reversal

> The directionality and its reversal were absent for other areas such as lateral prefrontal
> cortex, whose net flow had a lower median and higher variability (p = 0.16, paired permutation
> test, correct trials).

A claim of absence: per participant ΔGranger [4 8] Hz to the DLPFC virtual sensor, reading d1
(the code's), all correct trials of the GC method (t1), paired permutation test with 200 sign
flips, for each hippocampal reading. The other areas and readings are shown as variants.

In [ ]:
rng = np.random.default_rng(101)
rows = []
for area in ["DLPFC", "OFC", "PPC", "V1", "AC"]:
    for rd in READINGS:
        v = VAL[rd][area]
        rows.append(dict(area=area, reading=rname(rd), enc_median=v.enc.median(), maint_median=v.maint.median(),
                         median_diff=(v.maint - v.enc).median(), sd_enc=v.enc.std(), sd_maint=v.maint.std(),
                         p200=wmflow.paired_signflip(v.maint.values, v.enc.values, 200, rng)))
fe = pd.DataFrame(rows)
main = fe[(fe.area == "DLPFC") & fe.reading.str.contains("-t1-d1")]
print("main test (DLPFC, t1, d1):"); print(main.round(4).to_string(index=False))
print("\nall readings, DLPFC: p < 0.05 in", int((fe[fe.area == "DLPFC"].p200 < 0.05).sum()), "of", len(READINGS))
print(fe.pivot_table(index="reading", columns="area", values="p200").round(3).to_string())

**Result: reproduced.** By the paper's test on the DLPFC source (d1, all trials), encoding and
maintenance ΔGranger do not differ: p = 0.065 (h1), 0.27 (h2) and 0.060 (h3), with median
differences of 0.003 to 0.006. Under the d2 readings every area, DLPFC included, shows the
"reversal" at p = 0.005, so the contrast between the auditory source and the other areas that
the paper draws does not survive a common reading.

## fig4f-incorrect-no-reversal

> On incorrect trials, the net flow between auditory cortex and hippocampus did not have the same
> direction in all participants and did not reverse (p = 0.37, paired permutation test).

Blocked, `data_missing`. The test needs the auditory source on incorrect trials. The derivative
files hold the GC method's correct trials only (their trial counts equal the correct trials of set
sizes 6 and 8 in every participant, and the first 40 trials of each match the raw EEG of those
trials, data check above). The beamformer filters are not deposited, so the source cannot be
computed for other trials. The counts of incorrect trials that would have entered:

In [ ]:
inc = ev[wmflow.gc_trials(ev, correct=0)].groupby("sub").size().reindex(SUBS, fill_value=0)
print("incorrect trials of set size 6/8 without artifact, per participant:", inc.to_dict(), "| total", int(inc.sum()))
print("derivative trials that are incorrect trials: 0 (counts above equal the correct-trial counts in all", len(SUBS), "participants)")

## balanced-correct-subset

> With correct trials subsampled to the number of incorrect trials (10%), the reversal was equally
> present (p < 0.0005).

Reading t2 of fig4d-reversal-all: 10% of the correct trials, 200 draws, median over draws (the
Results) with the mean (the Methods) as the variant.

In [ ]:
b = d4[d4.reading.str.contains("-t2-")].copy()
b["criterion"] = (b.p200 < 0.05) & (b.both >= 13)
print(b[["reading", "enc_neg", "maint_pos", "both", "enc_median", "maint_median", "p200", "p_exact", "criterion"]].round(4).to_string(index=False))
print("t2 readings meeting the criterion:", int(b.criterion.sum()), "of", len(b), "| median aggregation:",
      int(b[b.reading.str.contains("median")].criterion.sum()), "| mean aggregation:", int(b[b.reading.str.contains("-mean-")].criterion.sum()))

**Result: reproduced, resting on reading d2.** With 10% of the correct trials and 200 draws, the
d2 readings give the paired test's smallest p (0.005 to 0.010; exact sign-flip p = 0.0001) and
the sign pattern in 13 to 15 participants (6 of 6 d2 readings, median and mean aggregation). The
d1 readings show no effect (1 to 4 participants, p = 0.38 to 0.86). The controls of
fig4d-reversal-all apply: d2 produces the pattern without a directed effect.

## Summary

| Claim | Outcome |
|---|---|
| cohort-sessions | partially reproduced (68 sessions, not 71; 15 sessions under 50 trials) |
| fig1b-accuracy | reproduced |
| capacity-cowan-k | reproduced |
| fig1c-response-time | reproduced (median, artifact-free correct trials) |
| fig2bc-gamma-encoding-p1 | partially reproduced (H3 over 100%, C2 not) |
| fig2d-cortical-beta-maintenance-p1 | reproduced |
| hipp-beta-maintenance | not reproduced |
| fig2fg-plv-theta-p1 | not reproduced |
| fig2h-gc-reversal-p1 | reproduced, resting on reading h1 and the per-frequency test |
| fig2ijk-spatial-spread-p1 | not reproduced |
| fig3-gc-reversal-p2-p3 | partially reproduced (Participant 3 only, reading h2) |
| fig3no-p4-right-no-gc | not reproduced (the test finds GC) |
| table1-left-hemisphere-sources | blocked: not runnable |
| fig4a-group-gc-reversal | not reproduced |
| scalp-temporal-laplacian | not reproduced |
| fig4bc-auditory-strongest | not reproduced |
| fig4d-reversal-all | not reproduced (pattern under d2, medians half the paper's) |
| table1-heschl-dgranger-correct | partially reproduced, resting on d2 (signs only) |
| fig4e-lpfc-no-reversal | reproduced |
| fig4f-incorrect-no-reversal | blocked: data missing |
| balanced-correct-subset | reproduced, resting on d2 |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The dataset holds 68 sessions, not 71, and the iEEG at 2000 to 4096 Hz; it is downsampled to
  500 Hz here, then to 40 Hz for GC, with scipy's polyphase filter in place of MATLAB's.
- GC is computed in Python (Hann-tapered FFT with 20 s padding, Wilson factorisation as in
  FieldTrip, Geweke's measure), checked against an analytic bivariate autoregressive process.
- The hippocampal contact and the white-matter reference are not given by the paper; three
  readings (h1 to h3) are run, with electrode and reference choices stated in the helper.
- The auditory source is the dataset's AC virtual sensor; the Heschl and superior temporal
  readings (a2, a3) cannot be run, as the dataset holds only five averaged virtual sensors.
- The PSD baseline drops fixation times whose 10-cycle window would start before the epoch.
- For time, the per-contact grid tests of fig2ijk and the spatial test run with h1 only, and
  the 10% subsampling uses 20 draws (the code's number) for the areas other than AC.

What was found about the dataset:

- participants.tsv maps the subjects to the paper's participant numbers (`eLife_identifier`),
  and the four ECoG participants are where the paper puts them. Three pathologies differ from
  Table 1 (Participants 5, 6 and 15).
- 68 sessions (3353 trials) against the paper's 71 sessions of 50 trials; 17 placeholder trials
  with set size 0 and no response; ProbeLetter missing in sub-10 to sub-15.
- channels.tsv gives low_cutoff 1000 and high_cutoff 0.5 Hz, the reverse of the sidecars'
  0.5 to 1000 Hz passband; the EDF headers leave the physical unit blank.
- In sub-11 the contacts named PHL lie at positive x (right) and those named PHR at negative x;
  in sub-01 the atlas labels name the opposite hemisphere to the MNI x coordinate for 12
  contacts.
- The LCMV derivative files hold no trial identifiers; their trials are the GC method's correct
  trials in order (checked against the raw EEG in every subject), with the artifact-flagged
  trials included in sub-13 and sub-14. They hold no incorrect trials and no beamformer filters.
- In the paper's code (calculateGranger_random_trial_sampling.m), the same row of the Granger
  spectrum is stored as cortex to hippocampus for encoding and as hippocampus to cortex for
  maintenance, and the per-participant value of Figure 4d is the minimum (encoding) or maximum
  (maintenance) over 4 to 8 Hz; this statistic yields the reported sign pattern without a
  directed effect.